use python/neuralnetmodelling/savecompletemodel.ipynb to export the model to keras
then  

conda activate guitarmidi_nn311
and 

ext/frugally-deep/keras_export/convert_model.py python/neuralnetmodelling/model_dir/guitarmidi-model-and-weights.keras python/neuralnetmodelling/model_dir/guitarmidi-model-and-weights.json

To convert the frugally deep json

In [1]:
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES,OUTPUT_DIM_ONSETS
import numpy as np
from model import build_cnn_model

print(INPUT_SHAPE)
cnn_model=build_cnn_model(INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.load_weights('guitarmidi.weights.h5')
# cnn_model.save('guitarmidi-model-and-weights.weights.h5')
cnn_model.save('model_dir/guitarmidi-model-and-weights.keras')
# cnn_model.export('model_dir/')

2025-10-07 06:07:01.331936: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-07 06:07:01.362139: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-07 06:07:01.928246: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


(312, 1)


I0000 00:00:1759810022.243378 3391684 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1080 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


In [20]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization

def fuse_bn_to_dense_or_conv(layer, bn_layer):
    # Get Conv/Dense weights and bias
    W, b = layer.get_weights()
    if b is None:
        b = np.zeros(W.shape[-1] if isinstance(layer, Dense) else layer.filters)
    
    # Get BN params
    gamma, beta, mean, var = bn_layer.get_weights()
    epsilon = bn_layer.epsilon
    
    # Fuse parameters
    std = np.sqrt(var + epsilon)
    W_fused = W * (gamma / std).reshape((-1,) if isinstance(layer, Dense) else (1, 1, 1, -1))
    b_fused = beta + (b - mean) * gamma / std
    
    return [W_fused, b_fused]

def fuse_model(model):
    # Create new model layers list
    new_layers = []
    skip_next = False
    
    for i in range(len(model.layers)):
        if skip_next:
            skip_next = False
            continue
        
        layer = model.layers[i]
        
        # Check if next layer is BatchNormalization
        if i + 1 < len(model.layers) and isinstance(model.layers[i + 1], BatchNormalization):
            bn_layer = model.layers[i + 1]
            if isinstance(layer, (Conv2D, Dense)):
                # Fuse
                fused_weights = fuse_bn_to_dense_or_conv(layer, bn_layer)
                layer.set_weights(fused_weights)
                new_layers.append(layer)
                skip_next = True  # skip BN layer
            else:
                new_layers.append(layer)
        else:
            new_layers.append(layer)

    # Build a new model without BN layers (only fused layers)
    inputs = model.inputs
    x = inputs[0]
    for layer in new_layers:
        x = layer(x)
    fused_model = tf.keras.Model(inputs=inputs, outputs=x)
    return fused_model


# Fuse BN layers into Conv2D/Dense weights
fused_model = fuse_model(cnn_model)

# Save the fused model for inference
fused_model.save('model_dir/guitarmidi-model-and-weights.keras')


In [ ]:
import tensorflow as tf
import numpy as np
import os

# --- Configuration ---
# NOTE: Replace 'original_model.h5' with the path to your actual trained model file.
MODEL_H5_PATH = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/model_dir/guitarmidi-model-and-weights.keras' 
OUTPUT_TFLITE_PATH = 'quantized_model.tflite'
INPUT_SHAPE = (256, 312, 1) # Must match the INPUT_HEIGHT/WIDTH/CHANNELS from your C++ code
# ---------------------

def representative_data_gen():
    """
    Generator function for the representative dataset.
    
    This is required for *full* integer quantization, where the converter needs to 
    sample the expected range of activation values.
    
    For demonstration, this yields 10 random samples matching the input shape.
    In a real application, replace this with actual, pre-processed training/validation data.
    """
    num_samples = 10
    
    # Note: Use a dtype of tf.float32, as Keras models expect float input.
    for _ in range(num_samples):
        # Create a dummy input tensor: (1, 256, 312, 1)
        data = np.random.rand(1, INPUT_SHAPE[0], INPUT_SHAPE[1], INPUT_SHAPE[2])
        yield [data.astype(np.float32)]

def convert_and_quantize(model_path, output_path, quantization_type='weights_only'):
    """
    Loads a Keras model and converts it to a quantized TFLite model.

    Args:
        model_path (str): Path to the input Keras model file (.h5 or SavedModel).
        output_path (str): Path to save the output TFLite file.
        quantization_type (str): 'weights_only' or 'full_integer'.
    """
    try:
        # 1. Load the Keras model
        print(f"Loading Keras model from: {model_path}")
        model = tf.keras.models.load_model(model_path)
    except Exception as e:
        print(f"Error loading model. Please ensure '{model_path}' exists.")
        print(e)
        return

    # 2. Create the TFLite Converter
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # 3. Apply Optimization (Quantization)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    if quantization_type == 'weights_only':
        # Option A: Weights-Only Quantization (Easiest, Good Speedup)
        # Weights (parameters) are converted to int8. Activations remain float32.
        print("\n--- Applying Weights-Only Quantization ---")
        
    elif quantization_type == 'full_integer':
        # Option B: Full Integer Quantization (Requires Representative Dataset, Max Speedup)
        # Weights and activations are converted to int8. Requires data for calibration.
        print("\n--- Applying FULL Integer Quantization (Maximum Speedup) ---")
        
        # Set the target data type for input and output to be int8 for max speed
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8  # Input is expected as int8
        converter.inference_output_type = tf.int8 # Output will be int8
        
        # Provide the representative dataset generator
        converter.representative_dataset = representative_data_gen
        
    else:
        print(f"Unknown quantization type: {quantization_type}")
        return

    # 4. Convert the model
    print("Converting model...")
    tflite_model = converter.convert()

    # 5. Save the TFLite model
    with open(output_path, 'wb') as f:
        f.write(tflite_model)

    print(f"\nSuccessfully saved quantized model to: {output_path}")

if __name__ == '__main__':
    # --- IMPORTANT ---
    # Start with 'weights_only' as it is simpler. If 5ms is still not met,
    # switch to 'full_integer' and ensure you provide real data in the 
    # representative_data_gen function.
    
    # Check if the placeholder model path exists before proceeding
    if not os.path.exists(MODEL_H5_PATH):
        print(f"ERROR: Model file not found at '{MODEL_H5_PATH}'.")
        print("Please ensure this path points to your trained Keras .h5 file.")
    else:
        # Choose one option below:
        convert_and_quantize(MODEL_H5_PATH, OUTPUT_TFLITE_PATH, quantization_type='weights_only')
        # convert_and_quantize(MODEL_H5_PATH, OUTPUT_TFLITE_PATH, quantization_type='full_integer') 
        
# Next step: Use Frugally-Deep's TFLite converter to get the final JSON:
# python convert_to_fdeep_json.py quantized_model.tflite guitarmidi-final-quantized.json
# (The exact FDeep TFLite converter script name may vary based on your version.)


ERROR: Model file not found at 'original_model.h5'.
Please ensure this path points to your trained Keras .h5 file.
